In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from delta.tables import *

In [0]:
bronze_accounts_df = spark.read.format("delta").table("bankaml.bronze.accounts")

accounts_quarantine_df = bronze_accounts_df.filter(
    col("account_id").isNull() 
    | col("customer_id").isNull() 
    | col("branch_id").isNull()
)

accounts_df = bronze_accounts_df.filter(
    col("account_id").isNotNull() 
    & col("customer_id").isNotNull() 
    & col("branch_id").isNotNull()
).dropDuplicates(["account_id"])

customers_silver_table_df = DeltaTable.forName(spark, "bankaml.silver.customers").toDF()

accounts_cust_joined_df = accounts_df.alias("s").join(
    customers_silver_table_df.alias("t"), 
    col("s.customer_id") == col("t.customer_id"), 
    "left"
)

valid_accounts = accounts_cust_joined_df.filter(col("t.customer_id").isNotNull()).select("s.*")
accounts_quarantine_df_2 = accounts_cust_joined_df.filter(col("t.customer_id").isNull()) \
    .select("s.*") \
    .withColumn("quarantine_reason", lit("customer_id not exist in bankaml.silver.customers"))

In [0]:
valid_accounts_df= valid_accounts.withColumns(
    {
        "account_type": upper(trim(col("account_type"))),
        "currency": upper(trim(col("currency"))),
        "open_date": col("open_date").cast(DateType()),
        "status": upper(coalesce(col("status"), lit("unknown"))),
        "created_at": col("created_at").cast(TimestampType()),
        "effective_start_date": col("created_at").cast(TimestampType()),
        "effective_end_date": lit("9999-12-31").cast(TimestampType()),
        "is_current": lit("Y").cast(StringType()),
        "updated_at": lit(None).cast(TimestampType()),
        "updated_by": lit(None).cast(StringType())
    }
)

In [0]:
%sql
create table if not exists bankaml.silver.accounts 
(
    account_id string,
    customer_id string, 
    account_type string, 
    currency string, 
    branch_id string, 
    open_date date, 
    status string,
    effective_start_date timestamp,
    effective_end_date timestamp,
    is_current string,
    created_at timestamp, 
    created_by string, 
    updated_at timestamp,
    updated_by string,
    _ingestion_ts timestamp
)
using delta;

In [0]:
accounts_silver_target = DeltaTable.forName(spark, "bankaml.silver.accounts")
accounts_target_df = accounts_silver_target.toDF()

account_target_joined_df = valid_accounts_df.alias("s").join(
    accounts_target_df.alias("t"),
    col("s.account_id") == col("t.account_id"),
    "left"
)

accounts_new_df = account_target_joined_df.filter(col("t.account_id").isNull()).select("s.*")
accounts_existing_df = account_target_joined_df.filter(col("t.account_id").isNull()).select("s.*")


In [0]:
(
    accounts_silver_target.alias("t").merge(
        accounts_existing_df.alias("s"),
        "t.account_id = s.account_id and t.is_current='Y'"
    )
    .whenMatchedUpdate(
        set={
            "effective_end_date": col("s.created_at").cast(TimestampType()),
            "is_current": lit("N"),
            "updated_at": current_timestamp(),
            "updated_by": lit("Databricks silver_accounts")
        }
    )
    .execute()
)

In [0]:
active_account_df = accounts_target_df.filter(col("is_current") == "Y")
changed_accounts_df = accounts_existing_df.alias("s") \
    .join(active_account_df.alias("t"), col("s.account_id") == col("t.account_id"), "left") \
        .filter(col("s.status") != col("t.status")).select("s.*")

accounts_insert_df = accounts_new_df.unionByName(changed_accounts_df)

In [0]:
accounts_insert_df.write.format("delta").mode("append").saveAsTable("bankaml.silver.accounts")

In [0]:
%sql
create table if not exists bankaml.quarantine.accounts
(
    account_id string,
    customer_id string,
    account_type string,
    currency string,
    branch_id string,
    open_date string,
    status string, 
    created_at STRING,
    created_by STRING,
    _ingestion_ts TIMESTAMP,
    quarantine_reason STRING,  
    quarantined_at TIMESTAMP,
    source_layer STRING
)
using delta;

In [0]:
accounts_quarantine_df = accounts_quarantine_df.withColumns(
    {
        "quarantine_reason": when(
            col("account_id").isNull() | col("customer_id").isNull() | col("branch_id").isNull(),
            lit("account_id/customer_id/branch_id is null")
        ),
        "quarantined_at": current_timestamp(),
        "source_layer": lit("silver")
    }
)

accounts_quarantine_df_2 = accounts_quarantine_df_2.withColumns(
    {
        "quarantined_at": current_timestamp(),
        "source_layer": lit("silver")
    }
)
quarantine_df = accounts_quarantine_df.unionByName(accounts_quarantine_df_2)

In [0]:
quarantine_df.write.format("delta").mode("append").saveAsTable("bankaml.quarantine.accounts")